In [1]:
from bert_score import score
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from torch.utils.data import Dataset
import random
import os
import json
from tqdm import tqdm
from peft import PeftModel
import bertviz        

import string


def normalize_text(text):
    """对文本进行标准化：移除标点、转为小写、去除多余空格"""
    text = text.lower().strip()
    text = ''.join(ch for ch in text if ch not in string.punctuation)  # 移除标点
    return ' '.join(text.split())  # 去掉多余空格

def calculate_em(pred, truth):
    """计算Exact Match (EM)"""
    pred = normalize_text(pred)
    truth = normalize_text(truth)
    return int(pred == truth)

def calculate_f1(pred, truth):
    """计算F1 Score"""
    pred_tokens = normalize_text(pred).split()
    truth_tokens = normalize_text(truth).split()

    common = set(pred_tokens) & set(truth_tokens)
    num_common = len(common)

    if num_common == 0:
        return 0.0, 0.0, 0.0

    precision = num_common / len(pred_tokens)
    recall = num_common / len(truth_tokens)
    f1 = 2 * (precision * recall) / (precision + recall)
    
    return precision, recall, f1

def compute_metrics(ground_truth, predicted):
    """
    计算整个数据集的平均 EM 和 F1 Score
    :param ground_truth: List of true answers
    :param predicted: List of predicted answers
    :return: Average EM and F1 Score
    """
    total_em = 0
    total_f1 = 0
    total_p = 0
    total_r = 0
    n = len(ground_truth)

    for truth, pred in zip(ground_truth, predicted):
        total_em += calculate_em(pred, truth)
        
        _p, _r, _f1 = calculate_f1(pred, truth)
        total_f1 += _f1
        total_p += _p
        total_r += _r

    avg_em = total_em / n
    avg_f1 = total_f1 / n
    avg_r = total_r / n
    avg_p = total_p / n
    return avg_em, avg_p, avg_r, avg_f1



2024-11-22 00:44:30.114467: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-22 00:44:30.114550: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-22 00:44:30.116461: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-22 00:44:30.128164: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
root_path = "./output/metrics/Qwen_final"

hotpot = {0:"hotpot_origin_ouput.jsonl", 1:"hotpot0_ouput.jsonl", 2:"hotpot1_ouput.jsonl"}
suqad = {0:"squad_origin_ouput.jsonl", 1:"squad0_ouput.jsonl", 2:"squad1_ouput.jsonl"}



key2ans = {}
for key, value in suqad.items():
    candidates = []
    answers=[]
    preds=[]
    print(value)
    perp = 0.0
    cnt = 0
    
    with open(os.path.join(root_path, value)) as f:
        for line in tqdm(f):
            example = json.loads(line)
#                 print(example)
            answer = example["answer"]
            pred = example["pred_answer"]
            while pred and pred[0] == " ":
                pred = pred[1:]
            
            while pred and pred[-1] == " ":
                pred = pred[:-1]
#             print(answer + "\t" + pred)
            
#             pred = pred.split("nswer:")
#             if len(pred) > 1:
#                 pred = pred[1]
#             else:
#                 pred = pred[0]

#             pred= pred.split("Please answer the question")
#             pred = pred[0]

#             pred =pred.split("You are a")
#             pred = pred[0]

            

            example["pred_answer"] = pred

            answers.append(answer)
            preds.append(pred)
            candidates.append(example)

#             p, r, f1 = score([pred], [answer], lang="en", verbose=False)

            new_key = example["question"] + "||"+ example["context"]
            if new_key not in key2ans:
                key2ans[new_key] = {}
            key2ans[new_key]["context"] = example["context"]
            key2ans[new_key]["question"] = example["question"]
            key2ans[new_key]["answer"]=answer
            example["perp"] = example["perp"]
            key2ans[new_key][key] = [pred, example["perp"]]
            perp += example["perp"]
            cnt +=1



    p, r, f1 = score(preds, answers, lang="en", verbose=True)
    
    avg_em, avg_p, avg_r, avg_f1 = compute_metrics(answers, preds)
    
    print()
    print("cnt:{}, F1:{:.4f}".format(len(f1), f1.mean()))
    print("cnt:{}, perp:{:.4f}".format(cnt, perp/cnt))
    print("Average EM: {:.4f}".format(avg_em))
    print("Average precise:{:.4f}, recall:{:.4f}, F1 Score: {:.4f}".format(avg_p, avg_r,avg_f1))
    for idx, _f1 in enumerate(f1):
        new_key = candidates[idx]["question"] + "||"+ candidates[idx]["context"]
        key2ans[new_key][key].append(_f1.numpy().item())
    #         print(key2ans[new_key])
        
with open(os.path.join(root_path, "_final.json"), "w") as fout:
    for key, value in key2ans.items():
        print(value["answer"], "\t", value[0], " ", value[1], " ", value[2])
        fout.write(json.dumps(value, ensure_ascii=False)+"\n")
    
    


    

            
            

hotpot_origin_ouput.jsonl


350it [00:00, 7857.61it/s]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/9 [00:00<?, ?it/s]

computing greedy matching.


...

calculating scores...
computing bert embedding.


  0%|          | 0/8 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/6 [00:00<?, ?it/s]

done in 1.92 seconds, 182.51 sentences/sec

cnt:350, F1:0.9274
cnt:350, perp:0.8327
Average EM: 0.4457
Average precise:0.6185, recall:0.5682, F1 Score: 0.5773
yes 	 ['Scott Derrickson and Ed Wood were both Americans.', 0.7025571465492249, 0.7986770868301392]   ['Scott Derrickson and Ed Wood were both Americans.', 0.7025571465492249, 0.7986770868301392]   ['no', 0.9203619360923767, 0.9954853653907776]
no 	 ['No', 0.3781655430793762, 0.9996763467788696]   ['No', 0.3781655430793762, 0.9996763467788696]   ['no', 0.9530998468399048, 0.9999999403953552]
YG Entertainment 	 ['YG Entertainment', 0.5331384539604187, 0.9999995827674866]   ['YG Entertainment', 0.5331384539604187, 0.9999995827674866]   ['YG Entertainment', 0.7671186923980713, 1.0000001192092896]
David Weissman 	 ['Nicolas Cage', 0.5490802526473999, 0.780475914478302]   ['Nicolas Cage', 0.5490802526473999, 0.780475914478302]   ['David Weissman', 0.8603194355964661, 1.0]
from 1986 to 2013 	 ['1903–1912', 0.6958828568458557, 0.8557235

In [3]:
# for idx, (key, value) in enumerate(key2ans.items()):
#     print(value[1])
#     print(value[0])
#     if idx > 1:
#         break
# #     if value[1][0] != value[0][0]:
        
        
    

In [4]:
with open("")

SyntaxError: expected ':' (810060329.py, line 1)